> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# IFR2 across tickers: which ones work, and does it beat the CDI?

The single-ticker notebook (`ifr2.ipynb`) asks which parameters work on one stock. This one asks which **tickers** work with a setup, and whether the result beats leaving the money in the CDI. The rules live in `docs/brief.md` section 7, the score in `docs/score.md` and the benchmark in `docs/cdi.md`. The notebook holds no trading logic.

**Before trusting any number here:**
- **Survivorship bias.** The list is today's most traded names. Names that were delisted or fell out of the index are not in it, which flatters a mean-reversion strategy. Liquid names reduce the problem but do not remove it.
- **Picking the best of many tickers is the same trap as picking the best of many parameters.** A ticker that looks great in-sample must show it out-of-sample too, and even then B3 stocks move together, so tickers are not independent samples.
- **Beating the CDI is a minimum, not proof.** The comparison ignores risk: read `excess` next to the drawdown and Calmar. Idle cash earns nothing in the strategy, and taxes and slippage are not modelled. Prices are raw (dividends and splits ignored).
- Daily candles, entry at the signal close, sizing by ATR with no stop loss: see `ifr2.ipynb` for what each of those means.

In [ ]:
import csv
import json
import urllib.request
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import pandas as pd
import yfinance as yf

from b3core.ifr2 import Ifr2Config
from b3core.optimizer import ParamRange, ParamSpace, Constraints, TrialCache, optimize
from b3core.strategy import Ifr2Strategy
from b3core.trialstore import SqliteTrialStore
from b3core.universe import rank_across_tickers, scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data: delete the folder to refresh
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Parameters

The 30 liquid names of the research prototype. Replace or extend the list as you like; each name needs at least three years of history to be used. `OOS_FRACTION` sets one shared out-of-sample date for every ticker: the date that leaves roughly the last 30% of the common history out. Everything is chosen on the in-sample part only.

In [ ]:
TICKERS = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
]
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive
OOS_FRACTION = 0.3
MIN_CANDLES_IS = 750  # about three years of daily candles

CONFIG = Ifr2Config.preset(
    rsi_entry=30, exit_mode="target", target_lookback=2, max_days=None,
    atr_period=5, atr_mult=2, max_risk_brl=2_000, capital_brl=100_000, lot_size=1,
)

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the cache folder. Tickers that fail to download or have too little history are listed, never silently dropped.

In [ ]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


loaded = {t: load_candles(t) for t in TICKERS}
missing = [t for t, c in loaded.items() if c is None]
short = [t for t, c in loaded.items() if c is not None and len(c) < MIN_CANDLES_IS]
data = {t: c for t, c in loaded.items() if c is not None and len(c) >= MIN_CANDLES_IS}
print(f"{len(data)} tickers loaded; no data: {missing or 'none'}; too short: {short or 'none'}")

## 3. One shared out-of-sample date

The union of all trading dates is cut at the point that leaves the last 30% out. Every ticker is split at that same date, so in-sample and out-of-sample mean the same calendar periods for everyone.

In [ ]:
local_dates = {
    t: pd.DatetimeIndex(c["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()
    for t, c in data.items()
}
all_dates = sorted(set().union(*local_dates.values()))
CUT = all_dates[int(len(all_dates) * (1 - OOS_FRACTION))]
in_sample = {t: c[(local_dates[t] < CUT)].reset_index(drop=True) for t, c in data.items()}
out_of_sample = {t: c[(local_dates[t] >= CUT)].reset_index(drop=True) for t, c in data.items()}
print(f"out-of-sample starts {CUT.date()}; in-sample {all_dates[0].date()} to {all_dates[all_dates.index(CUT) - 1].date()}")

## 4. The CDI

The benchmark is the daily CDI, compounded day by day over each ticker's own window and compared with the strategy's CAGR (`docs/cdi.md`). Only business days may be compounded, so the series is restricted to the trading days found in the candles.

Two sources, in this order:
1. The Central Bank's series SGS 12 (the CDI itself). Its API blocks some networks.
2. If that fails, the **Selic target rate** from Ipeadata (the government's economic data service), minus 0.10 points a year, which is where the CDI normally sits. This is an **approximation**: checked against the CDI of 2019 to 2023 it is within about 0.1 points a year. The cell prints which source it used.

To use the exact CDI, save series 12 as `samples/ifr2_universe/cdi.csv` with the columns `date` (dd/mm/yyyy) and `rate` (percent per business day).

In [ ]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(all_dates).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}; compounded over the whole period: {(1 + cdi / 100).prod():.2f}x")

## 5. Scan: one setup on every ticker

The same `CONFIG` on each ticker, in-sample and out-of-sample side by side. `excess` is the strategy's CAGR minus the CDI's over the same window, and `beats_cdi` is true when it is positive. Read `max_equity_drawdown` and `calmar` next to it: a ticker can beat the CDI and still carry a drawdown that is not worth it. Ranked by the in-sample `score_v1`.

In [ ]:
scan_is = scan_universe(in_sample, CONFIG, cdi, min_candles=MIN_CANDLES_IS // 2)
scan_oos = scan_universe(out_of_sample, CONFIG, cdi, min_candles=120)
columns = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1"]
scan = scan_is[columns].add_prefix("is_").join(scan_oos[columns].add_prefix("oos_"), how="left")
scan = scan.sort_values("is_score_v1", ascending=False)
print("skipped in-sample:", scan_is.attrs["skipped"] or "none", "| out-of-sample:", scan_oos.attrs["skipped"] or "none")
scan

How many tickers beat the CDI, and how many of those still do out-of-sample? A ticker that wins in-sample and loses out-of-sample was probably fitted by luck of the period.

In [ ]:
both = scan.dropna(subset=["oos_beats_cdi"])
summary = pd.Series(
    {
        "tickers": len(scan),
        "beat the CDI in-sample": int(scan["is_beats_cdi"].sum()),
        "beat the CDI out-of-sample": int(both["oos_beats_cdi"].sum()),
        "beat it in both": int((both["is_beats_cdi"] & both["oos_beats_cdi"]).sum()),
        "median in-sample excess": scan["is_excess"].median(),
        "median out-of-sample excess": both["oos_excess"].median(),
    },
    name="summary",
)
summary.to_frame()

## 6. Optimization: the best setup for each ticker

The scan above used one default setup on every ticker, so it says little about what IFR2 can do when tuned. Here every ticker gets the same grid as `ifr2.ipynb`, searched on its **in-sample** part only and ranked by `score_v1` (at least 30 trades). The best setting per ticker is then run on that ticker's **out-of-sample** part.

This is the most overfitted view in the notebook: each ticker picks its own best of hundreds of combinations, so its in-sample number is flattered by luck. The honest number is the out-of-sample one, and the next section asks whether one setting can work across tickers. Results are cached in a SQLite registry, so the run can be interrupted and resumed.

In [ ]:
SPACE = ParamSpace(
    {
        "rsi_entry": ParamRange(5, 40, 5),
        "exit_mode": ["target", "rsi", "target_or_rsi"],
        "target_lookback": [2, 3, 5, 7, 10],
        "rsi_exit": [60, 70, 80],
        "max_days": [None, 5, 10],
    },
    Ifr2Strategy(),
)
PARAMS = SPACE.names
MIN_TRADES = 30
WORKERS = 8
store = SqliteTrialStore(Path("../trials_ifr2_universe.sqlite"))  # git-ignored
cache = TrialCache(store)

trials_is = {}
for ticker, candles in in_sample.items():
    found = optimize(
        candles, CONFIG, SPACE, objective="score_v1", constraints=Constraints(min_trades=MIN_TRADES),
        oos_fraction=0.0, workers=WORKERS, cache=cache, run_label=f"universe {ticker} in-sample",
    )
    trials_is[ticker] = found.trials_table()
print(f"{len(trials_is)} tickers x {SPACE.restricted(CONFIG).size} combinations; backtests computed {cache.misses}, from the registry {cache.hits}")

In [ ]:
def config_from(row: pd.Series) -> Ifr2Config:
    """The grid parameters of a table row, applied on top of CONFIG."""
    values = {name: (None if pd.isna(row[name]) else row[name]) for name in PARAMS}
    return SPACE.build_config(CONFIG, values)


def run_setups(setups: dict[str, Ifr2Config]) -> pd.DataFrame:
    """Each ticker with its own setup: in-sample and out-of-sample side by side."""
    cols = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1"]
    rows = []
    for ticker, config in setups.items():
        a = scan_universe({ticker: in_sample[ticker]}, config, cdi, min_candles=1)[cols].add_prefix("is_")
        b = scan_universe({ticker: out_of_sample[ticker]}, config, cdi, min_candles=1)[cols].add_prefix("oos_")
        rows.append(a.join(b))
    return pd.concat(rows)


best_rows = {t: tbl[tbl["feasible"]].sort_values("objective", ascending=False).iloc[0]
             for t, tbl in trials_is.items() if tbl["feasible"].any()}
per_ticker = run_setups({t: config_from(r) for t, r in best_rows.items()})
per_ticker = pd.DataFrame({name: [best_rows[t][name] for t in per_ticker.index] for name in PARAMS},
                          index=per_ticker.index).join(per_ticker).sort_values("is_score_v1", ascending=False)
per_ticker[PARAMS + ["is_cagr", "is_excess", "is_score_v1", "oos_cagr", "oos_excess", "oos_beats_cdi", "oos_max_equity_drawdown", "oos_score_v1"]]

Compare the three views on the **out-of-sample** period, where nothing was chosen on the data: the default setup from section 5, each ticker's own best setting, and (next section) one setting shared by all tickers. If tuning per ticker does not beat the default out-of-sample, the in-sample gain was fitting.

## 7. One setting across tickers

A setting that works on many tickers is more likely to be an edge than one that works on a single ticker. Each combination is ranked by its **median** score across the tickers (one lucky ticker cannot carry it), with the share of tickers where it made money and the worst ticker. A combination must pass the 30-trade gate on at least 20 tickers to be ranked. Only the in-sample part is used.

In [ ]:
ranked = rank_across_tickers(trials_is, PARAMS, min_trades=MIN_TRADES, min_tickers=20)
ranked.head(10)

The top three shared settings, run on every ticker, in-sample and out-of-sample. `beat` counts the tickers that beat the CDI.

In [ ]:
def summarize(label: str, table: pd.DataFrame) -> dict:
    return {
        "setup": label,
        "tickers": len(table),
        "IS median cagr": table["is_cagr"].median(),
        "OOS median cagr": table["oos_cagr"].median(),
        "OOS median excess": table["oos_excess"].median(),
        "OOS beat CDI": int(table["oos_beats_cdi"].sum()),
        "OOS median equity dd": table["oos_max_equity_drawdown"].median(),
        "OOS median score": table["oos_score_v1"].median(),
    }


rows = [summarize("default (section 5)", scan),
        summarize("each ticker's own best", per_ticker)]
shared = {}
for rank in range(3):
    config = config_from(ranked.iloc[rank])
    table = run_setups({t: config for t in in_sample})
    shared[rank] = table
    label = "shared #%d: " % (rank + 1) + ", ".join(f"{n}={ranked.iloc[rank][n]}" for n in PARAMS)
    rows.append(summarize(label, table))
pd.DataFrame(rows).set_index("setup")

**How to read it.** The CDI comparison and the drawdown say whether a setup is worth running; the gap between the in-sample and out-of-sample columns says how much of it was fitting. If every setup loses to the CDI out-of-sample, the answer is not "tune harder": with one position at a time most capital sits idle, which is the portfolio question for the next section.

## 8. A portfolio of independent slots

One ticker at a time leaves most capital idle, so the next question is what happens when several tickers run side by side. The model is the simplest one: the capital is split equally into **slots**, one per ticker, and each slot runs IFR2 alone with `capital / N` and a risk budget of `max_risk_brl / N`. There is **no shared cash and no position limit**: a slot never borrows from another. The portfolio is the sum of the slot equity curves (`b3core.portfolio`), with the same equity drawdown and CDI comparison as a single ticker.

The study varies two things:
- **How many tickers (N)**: 1, 2, 5, 10, 15, 20, each as many random subsets of the loaded tickers as practical (`DRAWS`, seeded). The table shows the middle of the distribution (median) and its spread (10th and 90th percentile of CAGR), not one lucky subset.
- **How much risk**: `x1` keeps the total risk budget the same whatever N is, so the portfolio is the average of its tickers. `x3` risks three times as much (a bigger bet on the same signals). Without a stop loss this is nominal risk: a loss can exceed it.

Both windows are shown. No ticker is picked here, so in-sample and out-of-sample are both fair; their difference is the market regime, not fitting.

In [ ]:
from dataclasses import replace
from math import comb

import itertools

import numpy as np

from b3core.portfolio import PortfolioSummary, combine_equity, slot_equity, summarize_portfolio

CAPITAL = CONFIG.capital_brl
SIZES = [1, 2, 5, 10, 15, 20]
DRAWS = 300  # random subsets per size (all of them when there are fewer)
RISK_SCALES = {"same risk (x1)": 1.0, "three times the risk (x3)": 3.0}
names = sorted(set(in_sample) & set(out_of_sample))
windows = {"in-sample": in_sample, "out-of-sample": out_of_sample}


def slot_curves(data: dict, n: int, risk_scale: float) -> dict:
    """The equity curve of each ticker in a slot of CAPITAL / n, risking risk_scale x the budget / n."""
    config = replace(CONFIG, capital_brl=CAPITAL / n, max_risk_brl=CONFIG.max_risk_brl * risk_scale / n)
    return {t: slot_equity(data[t], config) for t in names}


def portfolio_of(curves: dict, members) -> PortfolioSummary:
    return summarize_portfolio(combine_equity({t: curves[t] for t in members}), CAPITAL, cdi)


rng = np.random.default_rng(0)  # seeded: the same subsets on every run
subsets = {}
for n in SIZES:
    if comb(len(names), n) <= DRAWS:
        subsets[n] = list(itertools.combinations(names, n))
    else:
        subsets[n] = [tuple(rng.choice(names, size=n, replace=False)) for _ in range(DRAWS)]

rows = []
random_runs = {}  # (window, scale, n) -> table with one row per subset
for window, data in windows.items():
    for scale_name, scale in RISK_SCALES.items():
        for n in SIZES:
            curves = slot_curves(data, n, scale)
            table = pd.DataFrame([portfolio_of(curves, s).__dict__ for s in subsets[n]])
            random_runs[window, scale_name, n] = table
            rows.append({
                "window": window, "risk": scale_name, "tickers": n, "subsets": len(table),
                "median cagr": table["cagr"].median(),
                "p10 cagr": table["cagr"].quantile(0.10),
                "p90 cagr": table["cagr"].quantile(0.90),
                "median equity dd": table["max_equity_drawdown"].median(),
                "median calmar": table["calmar"].median(),
                "median excess": table["excess"].median(),
                "beat CDI": table["beats_cdi"].mean(),
            })
random_study = pd.DataFrame(rows)
random_study.set_index(["window", "risk", "tickers"])

**How to read it.** Each row is a distribution over random subsets. Compare the rows of one window and risk setting as N grows: the spread (`p10` to `p90`) narrows because the average of more tickers is steadier, and `median equity dd` shows how much of the drawdown was ticker-specific and how much is the market. `median calmar` is CAGR over drawdown: if it barely moves, diversifying lowered the risk and the return together, which is not the same as improving the strategy. Raising the risk scales return and drawdown together, so it cannot turn a ticker that loses to the CDI into one that beats it by itself. `beat CDI` is the share of subsets with a positive excess.

Subsets of 15 and 20 out of the loaded tickers overlap heavily, so the spread at large N is smaller than it would be for a larger universe.

## 9. Choosing the tickers on the in-sample part

Random subsets say what to expect on average. This section asks whether a **choice made in-sample** carries over: the tickers are ranked by their in-sample `score_v1` with the default `CONFIG` (section 5), the best N form a portfolio, and the portfolio is judged out-of-sample. The worst N are shown as a control: if the ranking means anything, they should do worse.

`beats % of random` is the share of random subsets of the same size with a lower out-of-sample CAGR. A value near 50% means the choice is no better than chance; near 100% means it persisted. It is inflated at large N, where the chosen set and the random sets share most of their tickers.

In [ ]:
# Layer 2: pick tickers on the in-sample part only, judge the portfolio on the out-of-sample part.
def judge(label: str, members, scale_name: str = "same risk (x1)") -> dict:
    """The out-of-sample portfolio of `members`, and where it falls among random subsets of its size."""
    n = len(members)
    curves = slot_curves(out_of_sample, n, RISK_SCALES[scale_name])
    summary = portfolio_of(curves, members)
    if n in SIZES:
        random_cagr = random_runs["out-of-sample", scale_name, n]["cagr"]
    else:  # a size we did not draw: draw it now
        draws = [tuple(rng.choice(names, size=n, replace=False)) for _ in range(DRAWS)]
        random_cagr = pd.Series([portfolio_of(curves, d).cagr for d in draws])
    return {
        "selection": label, "tickers": n, "oos cagr": summary.cagr, "oos equity dd": summary.max_equity_drawdown,
        "oos calmar": summary.calmar, "oos excess": summary.excess, "beats CDI": summary.beats_cdi,
        "beats % of random": float((random_cagr < summary.cagr).mean()),
    }


by_score = scan["is_score_v1"].dropna().sort_values(ascending=False).index
selected = pd.DataFrame(
    [judge("top by in-sample score", list(by_score[:n])) for n in SIZES]
    + [judge("bottom by in-sample score", list(by_score[-n:])) for n in SIZES]
)
selected

## 10. Clusters

Three ways to group tickers, all built from the **in-sample** part only and judged out-of-sample like the section above.

- **Behaviour clusters (KMeans):** tickers grouped by what IFR2 did on them in-sample (CAGR, win rate, profit factor, equity drawdown, trade count) and by how much the stock moves. These features include the in-sample result, so this is a cousin of section 9, not an independent test.
- **Correlation clusters:** tickers grouped by how their prices move together (complete-linkage hierarchical clustering on `1 - correlation` of daily returns). The idea is that tickers from different clusters should not fall at the same time. B3 stocks are strongly tied to the market, so expect one large group and a few smaller ones.
- **Diversified picks:** start from the best in-sample ticker and keep adding the ticker whose in-sample **strategy** returns correlate least with those already chosen, among the better half by score. This is the "tickers that compensate each other" idea, measured on the strategy's equity and not on prices.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

K_CLUSTERS = 3
K_CORRELATION = 4

# 2a. Behaviour clusters: what the strategy did on each ticker in-sample, plus how the stock moves.
volatility = pd.Series({
    t: float(np.log(in_sample[t]["close"]).diff().std()) for t in names
})
features = scan_is.loc[names, ["cagr", "win_rate", "profit_factor", "max_equity_drawdown", "n_trades"]].assign(
    volatility=volatility
)
features = features.replace([np.inf, -np.inf], np.nan).fillna(features.median())
scaled = StandardScaler().fit_transform(features)
kmeans = KMeans(n_clusters=K_CLUSTERS, n_init=10, random_state=0).fit(scaled)
behaviour = pd.Series(kmeans.labels_, index=features.index, name="cluster")
profile = features.groupby(behaviour).median().assign(tickers=behaviour.value_counts().sort_index())
profile.join(behaviour.groupby(behaviour).apply(lambda s: ", ".join(s.index)).rename("members"))

In [ ]:
# 2b. Correlation clusters: tickers whose prices move together, from in-sample daily returns.
closes = pd.DataFrame({t: in_sample[t].set_index("ts")["close"] for t in names})
corr = np.log(closes).diff().corr()
distance = squareform((1 - corr).to_numpy(), checks=False)
corr_labels = fcluster(linkage(distance, method="complete"), t=K_CORRELATION, criterion="maxclust")
correlated = pd.Series(corr_labels, index=corr.index, name="cluster")
correlated.groupby(correlated).apply(lambda s: ", ".join(s.index)).rename("members").to_frame().assign(tickers=correlated.value_counts().sort_index())

Each group as an equal-split portfolio, out-of-sample. `beats % of random` compares it with random subsets of the same size.

In [ ]:
# 2c. Diversified picks: start from the best in-sample score and add the ticker whose in-sample
# strategy returns correlate least with those already chosen, among the better half by score.
is_curves = slot_curves(in_sample, 1, 1.0)
strategy_returns = pd.DataFrame({t: c.diff() for t, c in is_curves.items()}).ffill().fillna(0.0)
strategy_corr = strategy_returns.corr()
pool = list(by_score[: len(by_score) // 2])


def diversified(n: int) -> list[str]:
    chosen = [pool[0]]
    while len(chosen) < n:
        rest = [t for t in pool if t not in chosen]
        chosen.append(min(rest, key=lambda t: strategy_corr.loc[t, chosen].mean()))
    return chosen


rows = [judge(f"behaviour cluster {c}", list(behaviour.index[behaviour == c])) for c in sorted(behaviour.unique())]
rows += [judge(f"correlation cluster {c}", list(correlated.index[correlated == c])) for c in sorted(correlated.unique())]
best_in_cluster = [by_score[by_score.isin(correlated.index[correlated == c])][0] for c in sorted(correlated.unique())]
rows.append(judge("best of each correlation cluster", best_in_cluster))
rows += [judge("diversified by strategy correlation", diversified(n)) for n in (2, 5, 10)]
pd.DataFrame(rows)

**Caveats for sections 8 to 10.**
- **Few tickers, strongly correlated.** There are only about 28 names, mostly in one market block. A cluster of two or three tickers is an anecdote, not a finding.
- **Selection on in-sample results is selection.** Sections 9 and 10 pick by in-sample performance, so the out-of-sample column is the only honest one, and it is one regime (a high-Selic period) on the same stocks. A result that holds there is a lead to test on new data, not proof.
- **The ticker list is a survivor list** (today's most traded names), which flatters mean reversion. Costs beyond the 0.03% fee, taxes and slippage are not modelled, and the CDI is the Selic-target approximation when the Central Bank API is blocked.
- **Beating the CDI is a minimum.** Read the drawdown and Calmar next to every excess figure.

## 11. Ticker k-fold: does a setting work on tickers it never saw?

Section 7 chose one shared setting using **all** the tickers, so every ticker helped pick the setting that is then tested on it. Ticker k-fold removes that: the tickers are dealt into `K_FOLDS` groups; for each group, the shared setting is chosen on the **other** tickers only (the best median in-sample score, section 7's ranking) and then run on the hidden group. The setting never saw those tickers.

Two readings of the hidden group, from weaker to stronger:
- **In-sample window (IS columns):** unseen tickers, but the same calendar period the setting was chosen on. Stocks move together, so the period is still partly known. Treat it as optimistic.
- **Out-of-sample window (OOS columns):** unseen tickers **and** unseen time. This is the number to trust.

The whole split is repeated `REPEATS` times with different random folds, so no single lucky split decides the answer. The default `CONFIG` (no setting chosen) is the baseline: a chosen setting is only worth having if it beats it on the OOS columns.

In [ ]:
K_FOLDS = 5
REPEATS = 5  # a fresh random split each time, so no single split decides the answer
MIN_SHARE = 0.7  # a setting must pass the trade gate on 70% of the training tickers


def ticker_folds(tickers: list[str], k: int, seed: int) -> list[list[str]]:
    """Shuffle the tickers (seeded) and deal them into k folds of nearly equal size."""
    order = list(np.random.default_rng(seed).permutation(tickers))
    return [sorted(order[i::k]) for i in range(k)]


held_rows = []  # one row per ticker and repeat: the held-out result with the setting chosen without it
chosen = []  # the setting each fold picked, to see whether the folds agree
for repeat in range(REPEATS):
    for fold, held_out in enumerate(ticker_folds(sorted(trials_is), K_FOLDS, seed=repeat)):
        train = {t: tbl for t, tbl in trials_is.items() if t not in held_out}
        ranked_train = rank_across_tickers(
            train, PARAMS, min_trades=MIN_TRADES, min_tickers=int(MIN_SHARE * len(train))
        )
        best = ranked_train.iloc[0]
        chosen.append({"repeat": repeat, "fold": fold, **{n: best[n] for n in PARAMS},
                       "train median score": best["median_objective"]})
        outcome = run_setups({t: config_from(best) for t in held_out})
        held_rows.append(outcome.assign(repeat=repeat, fold=fold))
held_out_all = pd.concat(held_rows)
chosen = pd.DataFrame(chosen)
chosen

The table above shows which setting each fold picked. If the folds keep picking the same region, the choice is stable and does not depend on which tickers were hidden; if they jump around, the ranking is mostly noise.

In [ ]:
def view(label: str, table: pd.DataFrame) -> dict:
    return {
        "setup": label,
        "IS median cagr": table["is_cagr"].median(),
        "IS median score": table["is_score_v1"].median(),
        "OOS median cagr": table["oos_cagr"].median(),
        "OOS median excess": table["oos_excess"].median(),
        "OOS beat CDI": int(table["oos_beats_cdi"].sum()),
        "OOS median equity dd": table["oos_max_equity_drawdown"].median(),
        "OOS median score": table["oos_score_v1"].median(),
        "profitable OOS": float((table["oos_result"] > 0).mean()),
    }


per_repeat = pd.DataFrame(
    [view(f"fold-chosen, repeat {r}", g) for r, g in held_out_all.groupby("repeat")]
).set_index("setup")
kfold = pd.concat(
    [
        per_repeat,
        per_repeat.mean().rename("fold-chosen, mean of repeats").to_frame().T,
        pd.DataFrame([view("default CONFIG (no choice made)", scan)]).set_index("setup"),
    ]
)
kfold

**How to read it.** Compare the "mean of repeats" row with the default row on the OOS columns: median CAGR, median score and the share of profitable tickers. A gain in the IS columns that disappears in the OOS columns is fitting to the period, not to the ticker. `OOS beat CDI` counts tickers, summed over the 28 held-out tickers of one repeat.

**Caveats.** The 28 tickers are not independent (one market block), so hiding some of them hides little of what the others know. Every repeat reuses the same 28 names and the same two periods, so the repeats are not independent evidence either: they show how much the split matters, not how many times the result was confirmed.